# Producer Compensation POC - Fabric Notebook Demo

**Scenario:** Ingest an Azure SQL DB `ProducerPayments` table into a Fabric Lakehouse with a **file-first** architecture:

1. **PaymentDetails** (JSON blob) → written as `.json` files in `Files/payment_details/`
2. **Attachment** (VARBINARY) → written as binary files in `Files/attachments/`
3. **Regular columns** + **file path pointers** → Delta table in `Tables/`

**Two demo patterns:**
1. **Notebook approach** — PySpark reads SQL → writes blobs to Files → creates Delta with pointers (this notebook)
2. **Pipeline approach** — Copy Activity + Notebook Activity (config at the end)

> **Prerequisites:** Run `01_create_table_and_seed.sql` on your Azure SQL Database first.

## 1. Setup — Connection Configuration (Entra ID Auth)

Uses **Microsoft Entra ID (passwordless)** authentication — no SQL username/password or Key Vault secrets needed.  
`mssparkutils.credentials.getToken()` fetches an Entra token scoped to Azure SQL, using the Fabric workspace identity or signed-in user.

> **Prerequisite:** The Fabric workspace managed identity (or your user) must be added as an Azure SQL DB user:  
> ```sql
> CREATE USER [<your-fabric-workspace-name>] FROM EXTERNAL PROVIDER;
> ALTER ROLE db_datareader ADD MEMBER [<your-fabric-workspace-name>];
> ```

In [ ]:
# ── Connection Config (Entra ID / Passwordless Auth) ──
# No username/password needed — uses the Fabric workspace identity (managed identity)
# or the signed-in user's Entra token to authenticate to Azure SQL DB.

server   = "<YOUR_SERVER>.database.windows.net"
database = "<YOUR_DB>"

# Get Entra ID access token for Azure SQL Database
access_token = mssparkutils.credentials.getToken("https://database.windows.net/")

jdbc_url = f"jdbc:sqlserver://{server}:1433;database={database};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30"
jdbc_properties = {
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "accessToken": access_token
}

# Table to ingest
source_table = "dbo.ProducerPayments"

# Silver Delta table (regular columns + file pointers only)
delta_table_silver = "producer_payments_silver"

# Lakehouse default mount path (available in all Fabric notebooks with attached Lakehouse)
lakehouse_base = "/lakehouse/default"

# File paths in Lakehouse Files section
payment_details_path = f"{lakehouse_base}/Files/payment_details"
attachments_path     = f"{lakehouse_base}/Files/attachments"

print("Config loaded (Entra ID auth — no secrets needed).")
print(f"  Server:          {server}")
print(f"  Database:        {database}")
print(f"  Source:          {source_table}")
print(f"  PaymentDetails → {payment_details_path}/")
print(f"  Attachments    → {attachments_path}/")

## 2. Read Source Table from Azure SQL DB

Read the `ProducerPayments` table via JDBC into a Spark DataFrame. This pulls all 6 columns including the `PaymentDetails` JSON blob and `Attachment` VARBINARY column.

In [ ]:
# Read from SQL Server via JDBC
df_raw = spark.read.jdbc(
    url=jdbc_url,
    table=source_table,
    properties=jdbc_properties
)

print(f"Rows read: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(truncate=False)

## 3. Write Blobs to Lakehouse Files Section

Extract `PaymentDetails` and `Attachment` from the DataFrame and write them as individual files.  
This avoids loading large blobs into Delta tables — keeping the Delta layer lean and fast.

In [ ]:
import json
import os

# Collect rows from the JDBC DataFrame (already in memory from cell 2)
rows = df_raw.collect()

# Create directories using native Python (no .crc files)
os.makedirs(payment_details_path, exist_ok=True)
os.makedirs(attachments_path, exist_ok=True)

print(f"Ready to process {len(rows)} rows.")

### 3a. Write PaymentDetails JSON Files

Write each `PaymentDetails` JSON blob as `Files/payment_details/{PaymentID}.json`.

In [ ]:
# ── Write PaymentDetails as individual JSON files (native Python I/O — no .crc files) ──
detail_paths = {}
for row in rows:
    pid = row["PaymentID"]
    payment_json = row["PaymentDetails"]
    
    if payment_json:
        file_name = f"{pid}.json"
        file_path = os.path.join(payment_details_path, file_name)
        
        # Pretty-format the JSON before writing
        formatted = json.dumps(json.loads(payment_json), indent=2)
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(formatted)
        
        detail_paths[pid] = f"Files/payment_details/{file_name}"
        print(f"  ✓ PaymentID {pid} → Files/payment_details/{file_name}")

print(f"\nWrote {len(detail_paths)} PaymentDetails files.")

In [ ]:
# ── Write Attachment binary as individual files (native Python I/O — no .crc files) ──
attachment_paths = {}
for row in rows:
    pid = row["PaymentID"]
    attachment_data = row["Attachment"]
    
    if attachment_data:
        # Decode the binary to get the metadata JSON, use it for the file name
        meta_str = attachment_data.decode("utf-8")
        meta = json.loads(meta_str)
        original_name = meta.get("file", f"{pid}.bin")
        
        file_name = f"{pid}_{original_name}"
        file_path = os.path.join(attachments_path, file_name)
        
        # Write the raw binary content
        with open(file_path, "wb") as f:
            f.write(attachment_data)
        
        attachment_paths[pid] = f"Files/attachments/{file_name}"
        print(f"  ✓ PaymentID {pid} → Files/attachments/{file_name}")
    else:
        attachment_paths[pid] = None
        print(f"  ○ PaymentID {pid} → No attachment (NULL)")

print(f"\nWrote {sum(1 for v in attachment_paths.values() if v)} Attachment files.")

## 5. Build Silver Delta Table with File Path Pointers

Create the Silver Delta table with **regular columns only** plus two pointer columns:
- `PaymentDetailsFilePath` → path to the `.json` file in `Files/payment_details/`
- `AttachmentFilePath` → path to the binary file in `Files/attachments/` (NULL if no attachment)

This keeps the Delta table lean while the actual blob data lives in the Files section.

In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DecimalType
from delta.tables import DeltaTable

# Build rows with regular columns + file pointers
silver_rows = []
for row in rows:
    pid = row["PaymentID"]
    silver_rows.append(Row(
        PaymentID=pid,
        ProducerID=row["ProducerID"],
        CarrierName=row["CarrierName"],
        PaymentDate=row["PaymentDate"],
        GrossAmount=float(row["GrossAmount"]),
        PaymentDetailsFilePath=detail_paths.get(pid),
        AttachmentFilePath=attachment_paths.get(pid)
    ))

df_silver = spark.createDataFrame(silver_rows)

# MERGE (upsert) into Silver Delta table
if not spark.catalog.tableExists(delta_table_silver):
    df_silver.write.format("delta").saveAsTable(delta_table_silver)
    print(f"Created Silver Delta table: {delta_table_silver}")
else:
    delta_target = DeltaTable.forName(spark, delta_table_silver)
    delta_target.alias("t").merge(
        df_silver.alias("s"),
        "t.PaymentID = s.PaymentID"
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    print(f"Merged into Silver Delta table: {delta_table_silver}")

# Show the result — note: no blob data, just pointers
print("\n=== Silver Table (regular cols + file pointers) ===")
spark.sql(f"SELECT * FROM {delta_table_silver} ORDER BY PaymentID").show(truncate=False)

## 6. Demo: Read File Content Back via Pointers

Prove the pointers work — read the Delta table, pick a row, follow its file path, and load the actual JSON/attachment content from the Files section.

In [ ]:
import json as json_lib

# Read Silver table
df_check = spark.table(delta_table_silver)

# For each row, resolve pointers and read file content
print("=== Resolving File Pointers ===\n")
for row in df_check.orderBy("PaymentID").collect():
    pid = row["PaymentID"]
    print(f"── PaymentID {pid} ({row['CarrierName']}, ${row['GrossAmount']}) ──")
    
    # Read PaymentDetails JSON file
    detail_fp = row["PaymentDetailsFilePath"]
    if detail_fp:
        full_path = os.path.join(lakehouse_base, detail_fp)
        with open(full_path, "r", encoding="utf-8") as f:
            parsed = json_lib.load(f)
        print(f"  PaymentDetails: {detail_fp}")
        print(f"    commission_type: {parsed['commission_type']}")
        print(f"    tier_level:      {parsed['tier_level']}")
        print(f"    product_lines:   {parsed['product_lines']}")
    
    # Read Attachment file
    attach_fp = row["AttachmentFilePath"]
    if attach_fp:
        full_path = os.path.join(lakehouse_base, attach_fp)
        with open(full_path, "rb") as f:
            meta = json_lib.loads(f.read().decode("utf-8"))
        print(f"  Attachment:       {attach_fp}")
        print(f"    file: {meta['file']}, pages: {meta['pages']}, size_kb: {meta['size_kb']}")
    else:
        print(f"  Attachment:       None")
    print()

## 6. Simulate Changes — Run SQL Updates/Inserts then Re-Ingest

Run `02_simulate_changes.sql` against your Azure SQL Database. It will:
- Insert 2 new rows (with and without attachments)
- Update GrossAmount + JSON blob + Attachment on PaymentID 2
- Update JSON blob only on PaymentID 1
- Add an attachment to PaymentID 3 (was NULL)

Then **re-run cells 2–5** to show:
- New `.json` / attachment files written to Files section
- Existing files overwritten with updated content
- Delta MERGE handles new + updated pointer rows

---

## 7. Demo 2: Full Pipeline Approach (Zero Notebook Dependency)

A **100% pipeline solution** — 3 activities, no notebook needed at all.

The trick: pointer columns are **computed in the SQL query** via `CONCAT()` + `CASE`, so the Copy Activity writes them directly into the Silver Delta table.

```
┌──────────────────┐
│ Lookup_BlobData  │  ← Fetch PaymentDetails + Attachment per row
│ (SQL → result)   │
└────────┬─────────┘
         │
┌────────▼──────────────────┐
│ ForEach_WriteBlobs        │  ← Parallel (batch=5)
│  ├─ Write .json file      │     Files/payment_details/{PaymentID}.json
│  └─ If attachment exists: │
│       Write .bin file     │     Files/attachments/{PaymentID}_attachment.bin
└────────┬──────────────────┘
         │
┌────────▼──────────────────────┐
│ Copy_SilverTable_WithPointers │  ← SQL query computes pointer columns
│ SQL → Silver Delta table      │     Regular cols + PaymentDetailsFilePath
│ (Overwrite)                   │                  + AttachmentFilePath
└───────────────────────────────┘
```

**Silver table source query (runs in the Copy Activity):**
```sql
SELECT PaymentID, ProducerID, CarrierName, PaymentDate, GrossAmount,
       CONCAT('Files/payment_details/', CAST(PaymentID AS VARCHAR(20)), '.json')
           AS PaymentDetailsFilePath,
       CASE WHEN Attachment IS NOT NULL
            THEN CONCAT('Files/attachments/', CAST(PaymentID AS VARCHAR(20)), '_attachment.bin')
            ELSE NULL
       END AS AttachmentFilePath
FROM dbo.ProducerPayments
```

In [ ]:
# Full pipeline definition — no notebook dependency
# See: pipeline/PL_ProducerComp_Ingest.json
import json

pipeline_summary = {
    "name": "PL_ProducerComp_Ingest",
    "total_activities": 3,
    "activities": {
        "1_Lookup_BlobData": {
            "type": "Lookup",
            "query": "SELECT PaymentID, PaymentDetails, CAST(Attachment AS NVARCHAR(MAX)) AS AttachmentStr FROM dbo.ProducerPayments",
            "firstRowOnly": False
        },
        "2_ForEach_WriteBlobs": {
            "type": "ForEach (parallel, batch=5)",
            "depends_on": "Lookup_BlobData",
            "inner": {
                "Write_PaymentDetails_JSON": "→ Files/payment_details/{PaymentID}.json",
                "If_HasAttachment": {
                    "condition": "@not(empty(item().AttachmentStr))",
                    "true": "Write_Attachment_File → Files/attachments/{PaymentID}_attachment.bin"
                }
            }
        },
        "3_Copy_SilverTable_WithPointers": {
            "type": "Copy",
            "depends_on": "ForEach_WriteBlobs",
            "query": "SELECT PaymentID, ProducerID, CarrierName, PaymentDate, GrossAmount, "
                     "CONCAT('Files/payment_details/', CAST(PaymentID AS VARCHAR(20)), '.json') AS PaymentDetailsFilePath, "
                     "CASE WHEN Attachment IS NOT NULL THEN CONCAT('Files/attachments/', CAST(PaymentID AS VARCHAR(20)), '_attachment.bin') ELSE NULL END AS AttachmentFilePath "
                     "FROM dbo.ProducerPayments",
            "sink": "producer_payments_silver (Overwrite)"
        }
    }
}

print(json.dumps(pipeline_summary, indent=2))
print("\n✓ Full pipeline JSON: pipeline/PL_ProducerComp_Ingest.json")
print("✓ No notebook required — all data movement handled by pipeline activities")

## 8. (Optional) Trigger Pipeline via REST API

Use Fabric REST API to trigger the pipeline programmatically — no notebook involved in the pipeline execution itself.

In [ ]:
# Trigger pipeline via Fabric REST API
import requests

workspace_id  = "<YOUR_WORKSPACE_ID>"
pipeline_id   = "<YOUR_PIPELINE_ID>"
fabric_token  = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")

response = requests.post(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{pipeline_id}/jobs/instances?jobType=Pipeline",
    headers={"Authorization": f"Bearer {fabric_token}", "Content-Type": "application/json"}
)

print(f"Pipeline trigger status: {response.status_code}")
if response.status_code == 202:
    print("Pipeline started! Check Fabric Monitor for progress.")
else:
    print(f"Response: {response.text}")

In [ ]:
# ── Final Validation ──

print("=== Silver Delta Table (regular cols + file pointers, no blobs) ===")
spark.sql(f"""
    SELECT PaymentID, ProducerID, CarrierName, PaymentDate, GrossAmount,
           PaymentDetailsFilePath, AttachmentFilePath
    FROM {delta_table_silver}
    ORDER BY PaymentID
""").show(truncate=False)

# List files using native Python (clean — no .crc files)
print("\n=== Files: payment_details/ ===")
for fname in sorted(os.listdir(payment_details_path)):
    fsize = os.path.getsize(os.path.join(payment_details_path, fname))
    print(f"  {fname}  ({fsize} bytes)")

print("\n=== Files: attachments/ ===")
for fname in sorted(os.listdir(attachments_path)):
    fsize = os.path.getsize(os.path.join(attachments_path, fname))
    print(f"  {fname}  ({fsize} bytes)")

# Delta history
print("\n=== Delta History (Silver) ===")
spark.sql(f"DESCRIBE HISTORY {delta_table_silver}").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)